# Phase 10B: Binary TabularGRPO for Fraud Classification

This notebook compares a supervised MoE Transformer with a binary, forced-action group-relative fine-tuning variant for transaction fraud classification: `Transaction features -> Legitimate or Fraud`.

It does not predict approve, verify, review, or decline decisions. Those actions should only be considered after a reliable fraud classifier exists.

## Experiment design

1. Use a chronological 60/20/20 train/validation/test split.
2. Train a supervised MoE Transformer warm-start.
3. Fine-tune a copy with binary group-relative rewards.
4. Include both binary actions in each group, and record group reward spread.
5. Compare model candidates using validation PR-AUC and choose each threshold on validation F1.
6. Select one final candidate using validation PR-AUC, then evaluate that candidate on test once.
7. Repeat training for multiple seeds.

> Research POC only. Synthetic-data results do not establish real banking performance.

## Simple mental model

For one transaction, the model has two possible answers: `0` (legitimate) and `1` (fraud). Correct predictions receive reward 1 and incorrect predictions receive reward 0.

Each group deliberately includes both actions, then adds actions sampled from the old policy. This guarantees reward variation, but the forced actions are not policy samples. Therefore this is a forced-action, GRPO-inspired variant, not an unbiased implementation of canonical GRPO; interpret comparisons accordingly.

In [ ]:
# 1. Mount Google Drive and configure paths
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import copy, hashlib, json, os, platform, random, subprocess, sys

REPO = 'https://github.com/deepak-malviyaa/enterprise-adaptive-learning-lab.git'
PROJECT_ROOT = Path('/content/enterprise-adaptive-learning-lab')
DRIVE_ROOT = Path('/content/drive/MyDrive/enterprise-adaptive-learning-lab-data')
RUNS_ROOT = DRIVE_ROOT / 'outputs' / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
USE_SYNTHETIC_DATA = True
DATASET_NAME = 'synthetic' if USE_SYNTHETIC_DATA else 'ulb_real'
DATA_PATH = (DRIVE_ROOT / 'synthetic' / 'synthetic_creditcard_compatible_100k.csv') if USE_SYNTHETIC_DATA else (DRIVE_ROOT / 'raw' / 'creditcard.csv')
print('Dataset:', DATA_PATH)

In [ ]:
# 2. Clone/update the project and install dependencies
if (PROJECT_ROOT / '.git').exists():
    result = subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], text=True, capture_output=True)
elif PROJECT_ROOT.exists():
    raise FileExistsError(f'{PROJECT_ROOT} exists but is not a Git checkout; inspect it or choose another PROJECT_ROOT.')
else:
    result = subprocess.run(['git', 'clone', REPO, str(PROJECT_ROOT)], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Git operation failed.')
if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)

In [ ]:
# 3. Imports and reproducibility
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score,
    f1_score, brier_score_loss, confusion_matrix, precision_recall_curve,
)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = [42, 123, 456]
print('Device:', DEVICE, 'Seeds:', SEEDS)

## Chronological data preparation

Training uses the earliest 60%, validation the next 20%, and test the latest 20%. Validation and test keep their natural class prevalence. The test partition is not used to select a model or threshold.

In [ ]:
# 4. Load, validate, split, and scale
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

DATASET_SHA256 = sha256(DATA_PATH)
df = pd.read_csv(DATA_PATH)
EXPECTED = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
missing = sorted(set(EXPECTED) - set(df.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')
if df[EXPECTED].isna().any().any():
    raise ValueError('Dataset contains missing values in required columns.')
if set(df['Class'].unique()) != {0, 1}:
    raise ValueError('Class must contain exactly the binary labels 0 and 1.')
if not np.isfinite(df[EXPECTED].to_numpy(dtype=np.float64)).all():
    raise ValueError('Required dataset columns must be finite numeric values.')
df = df.sort_values('Time', kind='mergesort').reset_index(drop=True)
a = int(len(df) * 0.60)
b = int(len(df) * 0.80)
train_df = df.iloc[:a].copy()
val_df = df.iloc[a:b].copy()
test_df = df.iloc[b:].copy()
for split_name, split in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    if set(split['Class'].unique()) != {0, 1}:
        raise ValueError(f'{split_name} split must contain both classes for the requested metrics.')
FEATURES = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']
Xtr = train_df[FEATURES].copy()
Xv = val_df[FEATURES].copy()
Xt = test_df[FEATURES].copy()
ytr = train_df['Class'].to_numpy(dtype=np.int64)
yv = val_df['Class'].to_numpy(dtype=np.int64)
yt = test_df['Class'].to_numpy(dtype=np.int64)
scaler = StandardScaler().fit(Xtr[['Time', 'Amount']])
for features in (Xtr, Xv, Xt):
    features.loc[:, ['Time', 'Amount']] = scaler.transform(features[['Time', 'Amount']])
Xtr = Xtr.to_numpy(dtype=np.float32)
Xv = Xv.to_numpy(dtype=np.float32)
Xt = Xt.to_numpy(dtype=np.float32)
VAL_X = torch.tensor(Xv, dtype=torch.float32, device=DEVICE)
TEST_X = torch.tensor(Xt, dtype=torch.float32, device=DEVICE)
print('Rows:', len(df), 'Frauds train/val/test:', int(ytr.sum()), int(yv.sum()), int(yt.sum()), 'Checksum:', DATASET_SHA256)

## Balanced training without double weighting

The training loader samples fraud more often. The supervised loss is ordinary cross-entropy, so fraud is not weighted twice. Validation and test keep natural prevalence.

In [ ]:
# 5. Balanced training loader factory
train_dataset = TensorDataset(torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.long))
class_counts = np.bincount(ytr, minlength=2)
if np.any(class_counts == 0):
    raise ValueError('Training split must contain both classes for balanced sampling.')
row_weights = (1.0 / class_counts)[ytr]
def make_loader(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    sampler = WeightedRandomSampler(
        torch.tensor(row_weights, dtype=torch.double),
        num_samples=len(ytr),
        replacement=True,
        generator=generator,
    )
    return DataLoader(train_dataset, batch_size=512, sampler=sampler)
print('Training class counts:', class_counts.tolist())

## MoE Transformer model

The model outputs two logits. Softmax converts them into legitimate and fraud probabilities. Router probabilities are retained for routing diagnostics.

In [ ]:
# 6. Model architecture
class Tokenizer(nn.Module):
    def __init__(self, n=30, d=48):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n, d) * 0.02)
        self.b = nn.Parameter(torch.zeros(n, d))
        self.ids = nn.Parameter(torch.randn(n, d) * 0.02)

    def forward(self, x):
        return x.unsqueeze(-1) * self.w + self.b + self.ids

class MoE(nn.Module):
    def __init__(self, d=48, e=4):
        super().__init__()
        self.router = nn.Linear(d, e)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d, 96), nn.GELU(), nn.Linear(96, d))
            for _ in range(e)
        ])
        self.last = None

    def forward(self, x):
        probabilities = torch.softmax(self.router(x), dim=-1)
        self.last = probabilities
        values, indices = probabilities.topk(2, dim=-1)
        output = torch.zeros_like(x)
        for rank in range(2):
            for expert_index, expert in enumerate(self.experts):
                mask = indices[:, rank] == expert_index
                if mask.any():
                    output[mask] += values[mask, rank:rank + 1] * expert(x[mask])
        return output

    def balance_loss(self):
        mean_probability = self.last.mean(dim=0)
        return ((mean_probability - 1 / len(mean_probability)) ** 2).mean()

class MoETransformer(nn.Module):
    def __init__(self, input_dim=30, num_classes=2):
        super().__init__()
        dimension = 48
        self.tokenizer = Tokenizer(input_dim, dimension)
        self.cls = nn.Parameter(torch.zeros(1, 1, dimension))
        layer = nn.TransformerEncoderLayer(
            dimension, 4, 96, 0.1, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.moe = MoE(dimension, 4)
        self.head = nn.Linear(dimension, num_classes)

    def forward(self, x):
        tokens = self.tokenizer(x)
        encoded = self.encoder(torch.cat([self.cls.expand(len(x), -1, -1), tokens], dim=1))
        pooled = encoded[:, 0]
        return self.head(pooled + self.moe(pooled))

In [ ]:
# 7. Evaluation and threshold helpers
def evaluate_probabilities(labels, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(labels, probabilities)),
        'roc_auc': float(roc_auc_score(labels, probabilities)),
        'precision': float(precision_score(labels, predictions, zero_division=0)),
        'recall': float(recall_score(labels, predictions, zero_division=0)),
        'f1': float(f1_score(labels, predictions, zero_division=0)),
        'brier': float(brier_score_loss(labels, probabilities)),
        'threshold': float(threshold),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

def infer(model, features):
    model.eval()
    with torch.inference_mode():
        logits = model(features)
        probabilities = torch.softmax(logits, dim=-1)
        routing = model.moe.last.cpu().numpy()
    return probabilities[:, 1].cpu().numpy(), routing

def choose_threshold(labels, probabilities):
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    precision, recall = precision[:-1], recall[:-1]
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    best_index = np.lexsort((recall, f1))[-1]
    table = pd.DataFrame({
        'threshold': thresholds, 'precision': precision, 'recall': recall, 'f1': f1
    })
    return float(thresholds[best_index]), table

## Stage 1: supervised warm-start

The model learns ordinary fraud classification from verified labels. Early stopping and checkpoint selection use validation PR-AUC.

In [ ]:
# 8. Supervised training function
def supervised_warmstart(seed, epochs=25, patience=5):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    loader = make_loader(seed)
    model = MoETransformer().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    loss_function = nn.CrossEntropyLoss()
    best_score, best_state, stale = -1.0, None, 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for features, labels in loader:
            features, labels = features.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = loss_function(logits, labels) + 0.1 * model.moe.balance_loss()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * len(features)
        validation_probabilities, routing = infer(model, VAL_X)
        pr_auc = average_precision_score(yv, validation_probabilities)
        routing_entropy = float((-routing * np.log(routing + 1e-9)).sum(axis=1).mean())
        history.append({
            'epoch': epoch, 'loss': total_loss / len(train_dataset),
            'val_pr_auc': float(pr_auc), 'routing_entropy': routing_entropy,
        })
        print('supervised', seed, epoch, 'PR-AUC', round(pr_auc, 4), 'routing H', round(routing_entropy, 4))
        if pr_auc > best_score:
            best_score, best_state, stale = pr_auc, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
        if stale >= patience:
            break
    if best_state is None:
        raise RuntimeError('Supervised training did not produce a valid checkpoint.')
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## Stage 2: binary forced-action group-relative fine-tuning

Every group contains both labels plus policy-sampled actions. Correct actions receive reward 1; incorrect actions receive reward 0. Since the forced pair makes zero-variance rewards impossible for binary labels, log reward spread instead of treating a zero-variance fraction as a useful health signal. Checkpoint selection uses validation PR-AUC.

In [ ]:
# 9. Binary group-relative fine-tuning
GROUP_SIZE = 10
CLIP = 0.2
KL_COEF = 0.01
ENTROPY_COEF = 0.01
BALANCE_COEF = 0.1
if GROUP_SIZE < 2:
    raise ValueError('GROUP_SIZE must be at least 2 to include both forced actions.')

def grpo_finetune(warm_model, seed, epochs=12, patience=4):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    loader = make_loader(seed)
    model = MoETransformer().to(DEVICE)
    model.load_state_dict(warm_model.state_dict())
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    best_score, best_state, stale = -1.0, None, 0
    history = []
    for epoch in range(1, epochs + 1):
        old_model = MoETransformer().to(DEVICE)
        old_model.load_state_dict(model.state_dict())
        old_model.eval()
        model.eval()
        total_loss, total_reward_std, seen = 0.0, 0.0, 0
        last_kl = last_entropy = 0.0
        for features, labels in loader:
            features, labels = features.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                old_logits = old_model(features)
                old_probabilities = torch.softmax(old_logits, dim=-1)
                forced_actions = torch.tensor([0, 1], device=DEVICE).unsqueeze(0).expand(len(features), -1)
                sampled_actions = torch.multinomial(old_probabilities, GROUP_SIZE - 2, replacement=True)
                actions = torch.cat([forced_actions, sampled_actions], dim=1)
                old_selected = old_probabilities.gather(1, actions)
                rewards = (actions == labels.unsqueeze(1)).float()
                reward_std = rewards.std(dim=1, keepdim=True, correction=0)
                advantages = (rewards - rewards.mean(dim=1, keepdim=True)) / (reward_std + 1e-8)
                total_reward_std += reward_std.sum().item()
                seen += len(features)
            current_probabilities = torch.softmax(model(features), dim=-1)
            current_selected = current_probabilities.gather(1, actions)
            ratio = current_selected / old_selected.clamp_min(1e-8)
            unclipped = ratio * advantages
            clipped = torch.clamp(ratio, 1 - CLIP, 1 + CLIP) * advantages
            surrogate = torch.where(advantages >= 0, torch.minimum(unclipped, clipped), torch.maximum(unclipped, clipped))
            policy_loss = -surrogate.mean()
            old_safe = old_probabilities.clamp_min(1e-8)
            current_safe = current_probabilities.clamp_min(1e-8)
            kl = (old_safe * (old_safe.log() - current_safe.log())).sum(dim=1).mean()
            entropy = -(current_safe * current_safe.log()).sum(dim=1).mean()
            loss = policy_loss + KL_COEF * kl - ENTROPY_COEF * entropy + BALANCE_COEF * model.moe.balance_loss()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item()
            last_kl, last_entropy = float(kl.item()), float(entropy.item())
        validation_probabilities, routing = infer(model, VAL_X)
        pr_auc = average_precision_score(yv, validation_probabilities)
        routing_entropy = float((-routing * np.log(routing + 1e-9)).sum(axis=1).mean())
        history.append({
            'epoch': epoch, 'loss': total_loss / len(loader),
            'val_pr_auc': float(pr_auc),
            'mean_group_reward_std': total_reward_std / seen,
            'kl': last_kl, 'policy_entropy': last_entropy,
            'routing_entropy': routing_entropy,
        })
        print('group-relative', seed, epoch, 'PR-AUC', round(pr_auc, 4), 'reward SD', round(total_reward_std / seen, 4), 'KL', round(last_kl, 4))
        if pr_auc > best_score:
            best_score, best_state, stale = pr_auc, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
        if stale >= patience:
            break
    if best_state is None:
        raise RuntimeError('Group-relative training did not produce a valid checkpoint.')
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## Run multiple seeds

All candidate models and thresholds are compared on validation data. No test predictions are made in this loop.

In [ ]:
# 10. Train both stages for every seed; retain validation results only
runs = []
for seed in SEEDS:
    supervised_model, supervised_history = supervised_warmstart(seed)
    supervised_val, _ = infer(supervised_model, VAL_X)
    supervised_threshold, supervised_thresholds = choose_threshold(yv, supervised_val)
    supervised_val_metrics = evaluate_probabilities(yv, supervised_val, supervised_threshold)
    grpo_model, grpo_history = grpo_finetune(supervised_model, seed)
    grpo_val, _ = infer(grpo_model, VAL_X)
    grpo_threshold, grpo_thresholds = choose_threshold(yv, grpo_val)
    grpo_val_metrics = evaluate_probabilities(yv, grpo_val, grpo_threshold)
    runs.append({
        'seed': seed,
        'supervised_model': supervised_model,
        'supervised_history': supervised_history,
        'supervised_val_metrics': supervised_val_metrics,
        'supervised_thresholds': supervised_thresholds,
        'grpo_model': grpo_model,
        'grpo_history': grpo_history,
        'grpo_val_metrics': grpo_val_metrics,
        'grpo_thresholds': grpo_thresholds,
    })
    print('seed', seed, 'supervised validation', supervised_val_metrics, 'group-relative validation', grpo_val_metrics)

In [ ]:
# 11. Compare validation performance across seeds
rows = []
for run in runs:
    rows.append({'seed': run['seed'], 'stage': 'supervised', **run['supervised_val_metrics']})
    rows.append({'seed': run['seed'], 'stage': 'binary_group_relative', **run['grpo_val_metrics']})
validation_results = pd.DataFrame(rows)
display(validation_results)
validation_summary = validation_results.groupby('stage').agg({
    'pr_auc': ['mean', 'std'], 'roc_auc': ['mean', 'std'],
    'precision': ['mean', 'std'], 'recall': ['mean', 'std'],
    'f1': ['mean', 'std'], 'fp': ['mean', 'std'], 'fn': ['mean', 'std'],
})
display(validation_summary)

## Select and evaluate one final candidate

Select the candidate with the highest validation PR-AUC across both training stages and all seeds. Its threshold was selected on validation F1. Only this one frozen candidate is evaluated on the test partition. This preserves the test set for a single final estimate, though the estimate still reflects the stated chronological split and dataset.

In [ ]:
# 12. Select by validation PR-AUC; evaluate test exactly once
candidates = []
for run in runs:
    candidates.append({
        'seed': run['seed'], 'stage': 'supervised',
        'model': run['supervised_model'],
        'metrics': run['supervised_val_metrics'],
        'thresholds': run['supervised_thresholds'],
    })
    candidates.append({
        'seed': run['seed'], 'stage': 'binary_group_relative',
        'model': run['grpo_model'],
        'metrics': run['grpo_val_metrics'],
        'thresholds': run['grpo_thresholds'],
    })
best_candidate = max(candidates, key=lambda candidate: candidate['metrics']['pr_auc'])
BEST_MODEL = best_candidate['model']
BEST_SEED = best_candidate['seed']
BEST_STAGE = best_candidate['stage']
BEST_THRESHOLD = best_candidate['metrics']['threshold']
best_test_probabilities, test_routing = infer(BEST_MODEL, TEST_X)
BEST_TEST_METRICS = evaluate_probabilities(yt, best_test_probabilities, BEST_THRESHOLD)
routing_utilization = test_routing.mean(axis=0)
routing_entropy = float((-test_routing * np.log(test_routing + 1e-9)).sum(axis=1).mean())
print('Selected stage:', BEST_STAGE, 'seed:', BEST_SEED)
print('Validation metrics:', json.dumps(best_candidate['metrics'], indent=2))
print('Single final test evaluation:', json.dumps(BEST_TEST_METRICS, indent=2))
print('Mean router probabilities:', routing_utilization)
print('Routing entropy:', routing_entropy)

In [ ]:
# 13. Save models, histories, validation results, and final test result
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + f'_binary_tabulargrpo_{DATASET_NAME}'
RUN_DIR = RUNS_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
for run in runs:
    seed = run['seed']
    torch.save(run['supervised_model'].state_dict(), RUN_DIR / f'supervised_seed_{seed}.pt')
    torch.save(run['grpo_model'].state_dict(), RUN_DIR / f'binary_group_relative_seed_{seed}.pt')
    run['supervised_history'].to_csv(RUN_DIR / f'supervised_history_seed_{seed}.csv', index=False)
    run['grpo_history'].to_csv(RUN_DIR / f'group_relative_history_seed_{seed}.csv', index=False)
    run['supervised_thresholds'].to_csv(RUN_DIR / f'supervised_thresholds_seed_{seed}.csv', index=False)
    run['grpo_thresholds'].to_csv(RUN_DIR / f'group_relative_thresholds_seed_{seed}.csv', index=False)
validation_results.to_csv(RUN_DIR / 'validation_results.csv', index=False)
validation_summary.to_csv(RUN_DIR / 'validation_summary.csv')
final_result = {
    'selected_stage': BEST_STAGE, 'selected_seed': BEST_SEED,
    'validation_metrics': best_candidate['metrics'],
    'test_metrics': BEST_TEST_METRICS,
    'dataset_name': DATASET_NAME, 'dataset_sha256': DATASET_SHA256,
    'group_size': GROUP_SIZE, 'clip_epsilon': CLIP,
    'kl_coef': KL_COEF, 'entropy_coef': ENTROPY_COEF,
    'balance_coef': BALANCE_COEF, 'seeds': SEEDS,
    'mean_router_probabilities': routing_utilization.tolist(),
    'routing_entropy': routing_entropy,
    'warning': 'Research POC only; forced-action group-relative variant, not canonical GRPO.',
}
torch.save(BEST_MODEL.state_dict(), RUN_DIR / 'selected_model.pt')
(RUN_DIR / 'final_result.json').write_text(json.dumps(final_result, indent=2), encoding='utf-8')
print('Saved:', RUN_DIR)

## How to interpret the result

- Compare supervised and group-relative validation results across seeds; do not claim a GRPO improvement based on the test set used for selection.
- The selected final test result belongs to the single candidate chosen by validation PR-AUC.
- Review false positives, false negatives, threshold, and routing diagnostics in addition to PR-AUC.
- This experiment is a forced-action group-relative variant, and the synthetic-data result is not evidence of real-world banking performance.